In [ ]:
import pandas as pd
import numpy as np

def transformar_bronze_para_silver(path_bronze_lpr, path_bronze_criminal):

    df_lpr = pd.read_parquet(path_bronze_lpr)
    df_crim = pd.read_parquet(path_bronze_criminal)

    df_lpr = df_lpr.dropna(subset=['placa'])
    df_lpr = df_lpr[df_lpr['confianca_ocr'] >= 0.85]

    df_lpr['placa'] = df_lpr['placa'].str.replace('-', '').str.upper().str.strip()

    df_lpr = df_lpr.sort_values(by=['sensor_id', 'placa', 'timestamp'])
    df_lpr['diff_tempo'] = df_lpr.groupby(['sensor_id', 'placa'])['timestamp'].diff().dt.total_seconds()

    df_silver_lpr = df_lpr[(df_lpr['diff_tempo'] > 10) | (df_lpr['diff_tempo'].isna())]

    mapping_crimes = {'ROUBO DE AUTOMOVEL': 'ROUBO_VEICULO', 'FURTO': 'FURTO_VEICULO'}
    df_crim['tipo_crime'] = df_crim['ds_ocorrencia'].replace(mapping_crimes)

    df_final = pd.merge(
        df_silver_lpr,
        df_crim[['placa', 'tipo_crime', 'status_alerta']],
        on='placa',
        how='left'
    )

    return df_final